# 06D - XGBoost Classifier

Professional notebook for training and evaluating an XGBoost model.

## 1. Import Libraries

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
f1_score, roc_auc_score, classification_report,
ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

from xgboost import XGBClassifier

## 2. Load Dataset

In [ ]:
df=pd.read_csv("american_bankruptcy.csv")
df["target"]=df["status_label"].map({"alive":0,"failed":1})
display(df.head())

## 3. Prepare Features

In [ ]:
drop_cols=["status_label","target"]
if "company_name" in df.columns:
    drop_cols.append("company_name")
X=df.drop(columns=drop_cols)
y=df["target"]
num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns

## 4. Train/Test Split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,stratify=y,random_state=42)

## 5. Preprocessing

In [ ]:
preprocessor=ColumnTransformer([
("num",SimpleImputer(strategy="median"),num),
("cat",Pipeline([
("imp",SimpleImputer(strategy="most_frequent")),
("enc",OneHotEncoder(handle_unknown="ignore"))
]),cat)
])

## 6. Initialize XGBoost

In [ ]:
model=Pipeline([
("preprocessor",preprocessor),
("classifier",XGBClassifier(
n_estimators=300,
max_depth=6,
learning_rate=0.05,
subsample=0.8,
colsample_bytree=0.8,
eval_metric="logloss",
random_state=42
))
])

## 7. Train Model

In [ ]:
model.fit(X_train,y_train)

## 8. Cross Validation

In [ ]:
scores=cross_val_score(model,X_train,y_train,cv=5,scoring="f1")
print(scores)
print("Mean F1:",scores.mean())

## 9. Predictions

In [ ]:
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]

## 10. Evaluation

In [ ]:
metrics=pd.DataFrame({
"Metric":["Accuracy","Precision","Recall","F1","ROC-AUC"],
"Score":[accuracy_score(y_test,pred),
precision_score(y_test,pred),
recall_score(y_test,pred),
f1_score(y_test,pred),
roc_auc_score(y_test,proba)]
})
display(metrics)
print(classification_report(y_test,pred))

## 11. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()

## 12. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test,proba)
plt.show()

## 13. Precision-Recall Curve

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test,proba)
plt.show()

## 14. Feature Importance

In [ ]:
clf=model.named_steps["classifier"]
feat=list(num)
if len(cat)>0:
    feat.extend(model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["enc"].get_feature_names_out(cat))
imp=pd.DataFrame({"Feature":feat,"Importance":clf.feature_importances_})
display(imp.sort_values("Importance",ascending=False).head(20))

## 15. Save Model

In [ ]:
joblib.dump(model,"xgboost_model.joblib")

## 16. Executive Summary

In [ ]:
print("- XGBoost is a powerful gradient boosting algorithm.")
print("- Compare its performance with Logistic Regression, Decision Tree and Random Forest.")
print("- If it performs best, use it as the candidate for hyperparameter tuning.")